<a href="https://colab.research.google.com/github/pizzeman/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
clean = df.copy()

In [4]:

removed = clean.duplicated().sum()
clean.drop_duplicates(inplace=True)
print(f"Removed {removed} number")

log('duplicates', 'dropped exact duplicate rows', removed)

Removed 15 number
[duplicates] dropped exact duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [5]:
clean['price'] = (clean['price'].astype(str)
                  .str.replace('$', '').str.replace(',','')
                  .str.strip()
                  .astype(float))

assert clean['price'].dtype == float
log('price', 'price arrived as text. stripped price', clean['price'])

[price] price arrived as text. stripped price (0      12.0
1       7.5
2       7.5
3       7.5
4       7.5
       ... 
295     6.0
296     6.0
297     7.5
298     7.5
299    12.0
Name: price, Length: 300, dtype: float64 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [6]:
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = df['qty'].isnull().sum()
negative = (df['qty'] < 0).sum()

clean = clean[clean['qty'].notna() & (clean['qty'] > 0)].copy()
clean['qty'] = clean['qty'].astype(int)

log('qty', 'removed NaN quantities', missing)
log('qty', 'removed negative quantities', negative)

[qty] removed NaN quantities (13 row(s))
[qty] removed negative quantities (13 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [7]:
print('before', clean['item'].value_counts())

clean['item'] = clean['item'].str.lower()
clean['item'] = clean['item'].str.strip()
clean['item'] = clean['item'].str.replace('-', '')

ITEM_MAP = {
    'cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'foam finger': 'Foam Finger',
    'rain poncho': 'Rain Poncho',
}

clean['item'] = clean['item'].replace(ITEM_MAP)
print()
print('after', clean['item'].value_counts())
log('item', 'standardized item names', clean['item'])

before item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64

after item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64
[item] standardized item names (0       Rain Poncho
1       Foam Finger
2      Cheeseburger
4      Cheeseburger
5       Foam Finger
           ...     
294     Rain Poncho
295     Rain Poncho
297     Foam Finger
298     Rain Poncho
299    Cheeseburger
Name: item, Length: 275, dtype: object row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [8]:
print('before', sorted(clean['category'].unique()))

clean['category'] = clean['category'].str.lower()
clean['category'] = clean['category'].str.strip()
clean['category'] = clean['category'].str.replace('-', '')

CATEGORY_MAP = {
    'apparel': 'merch',
    'food': 'food',
    'merch': 'merch',
    'raingear': 'rain gear'
}
clean['category'] = clean['category'].replace(CATEGORY_MAP)

print('after: ', sorted(clean['category'].unique()))
log('category', 'standardized categories', clean['category'])

before [np.str_('Apparel'), np.str_('Food'), np.str_('Merch'), np.str_('RainGear'), np.str_('food'), np.str_('rain-gear')]
after:  ['food', 'merch', 'rain gear']
[category] standardized categories (0      rain gear
1          merch
2          merch
4          merch
5           food
         ...    
294        merch
295         food
297         food
298        merch
299         food
Name: category, Length: 275, dtype: object row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [9]:
assert clean.duplicated().sum() == 0
assert clean['qty'].min() >= 1
assert clean['price'].dtype == float
assert clean['item'].isin(ITEM_MAP.values()).all()
assert clean['category'].isin(CATEGORY_MAP.values()).all()

print('clean:', df.shape)

clean: (315, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [10]:
clean['revenue'] = clean['price'] * clean['qty']

print(clean.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2))
print('revenue:', clean['revenue'].sum())

category
food         1656.0
merch        1572.0
rain gear    1512.0
Name: revenue, dtype: float64
revenue: 4740.0


**What I would tell the vendor:** food appeared to have the highest revenue out of all the categories. Rain gear was high during this event, so it suggests that it was most likely raining.

Revenue after cleaning price -> \$4852.50
After removing duplicates -> \$4594.50
After cleaning qty -> \$4740.00
After consolidating item -> \$4740.00
After consolidating category -> \$4740.00

### TODO 8 — read back your log

In [11]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped exact duplicate rows,15
1,price,price arrived as text. stripped price,0 12.0 1 7.5 2 7.5 3 7....
2,qty,removed NaN quantities,13
3,qty,removed negative quantities,13
4,item,standardized item names,0 Rain Poncho 1 Foam Finger 2 ...
5,category,standardized categories,0 rain gear 1 merch 2 m...


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a. The largest change was after removing duplicate rows. First, price had to be converted into a float so that revenue could be calculated.
Before: revenue = \$4852.50 After removing duplicates -> \$4594.50

b. The choice of categories could be made differently. I chose to combine apparel and merch as teh same category, since I see apparel as a type of merch. If you did combine these, the total revenue would not change but the distribution of the revenue amongst the subcategories would. Merch would no longer have a higher revenue intake compared to rain gear.